# S24 — GANs

**Week 13 · Mon Nov 16, 2026 · Module 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s24_gans.ipynb)

Every cell below is a worked example from the [S24 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s24/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s24.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s24.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## Why training is a knife-edge


*Expected output starts with:* `step    0  D loss 1.3810  G loss 0.8556  near -2: 0.000  near +2: 0.000`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Real data: 1D mixture, half N(-2, 0.5^2), half N(+2, 0.5^2)
def real_batch(n):
    side = (torch.rand(n, 1) < 0.5).float() * 4 - 2
    return side + 0.5 * torch.randn(n, 1)

G = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))
D = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))
opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()
ones, zeros = torch.ones(128, 1), torch.zeros(128, 1)

for step in range(2001):
    # --- discriminator step ---
    x_real = real_batch(128)
    x_fake = G(torch.randn(128, 1)).detach()
    d_loss = bce(D(x_real), ones) + bce(D(x_fake), zeros)
    opt_d.zero_grad(); d_loss.backward(); opt_d.step()
    # --- generator step (non-saturating loss) ---
    x_fake = G(torch.randn(128, 1))
    g_loss = bce(D(x_fake), ones)
    opt_g.zero_grad(); g_loss.backward(); opt_g.step()

    if step % 250 == 0:
        with torch.no_grad():
            s = G(torch.randn(4000, 1))
        left = ((s + 2).abs() < 1).float().mean().item()
        right = ((s - 2).abs() < 1).float().mean().item()
        print(f"step {step:4d}  D loss {d_loss.item():.4f}  G loss {g_loss.item():.4f}  "
              f"near -2: {left:.3f}  near +2: {right:.3f}")

## Counting modes in 2D


*Expected output starts with:* `step    0  D loss 1.3226  G loss 0.6931  modes covered: 0/8`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Real data: mixture of 8 Gaussians on a ring of radius 2, std 0.15
angles = torch.arange(8) * (2 * torch.pi / 8)
centers = torch.stack([2 * torch.cos(angles), 2 * torch.sin(angles)], dim=1)  # (8, 2)

def real_batch(n):
    idx = torch.randint(0, 8, (n,))
    return centers[idx] + 0.15 * torch.randn(n, 2)

def modes_covered(samples):
    # a mode counts as covered if >= 2% of samples land within 0.45 of its center
    d = torch.cdist(samples, centers)           # (n, 8)
    nearest = d.argmin(dim=1)
    hit = d.min(dim=1).values < 0.45
    counts = torch.bincount(nearest[hit], minlength=8)
    return (counts.float() / len(samples) >= 0.02).sum().item(), counts.tolist()

G = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))
D = nn.Sequential(nn.Linear(2, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))
opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()
ones, zeros = torch.ones(256, 1), torch.zeros(256, 1)

for step in range(3001):
    x_real = real_batch(256)
    x_fake = G(torch.randn(256, 2)).detach()
    d_loss = bce(D(x_real), ones) + bce(D(x_fake), zeros)
    opt_d.zero_grad(); d_loss.backward(); opt_d.step()

    x_fake = G(torch.randn(256, 2))
    g_loss = bce(D(x_fake), ones)
    opt_g.zero_grad(); g_loss.backward(); opt_g.step()

    if step % 500 == 0:
        with torch.no_grad():
            m, counts = modes_covered(G(torch.randn(4000, 2)))
        print(f"step {step:4d}  D loss {d_loss.item():.4f}  G loss {g_loss.item():.4f}  "
              f"modes covered: {m}/8")

with torch.no_grad():
    m, counts = modes_covered(G(torch.randn(4000, 2)))
print(f"final per-mode sample counts (of 4000): {counts}")

## Why distance matters: from JS to Wasserstein


*Expected output starts with:* `    mu  distance  D(fake)  |sat grad|  |non-sat grad|  |W grad|`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Real data: a cluster at +2. "Generator": a cloud at trainable offset mu.
real = 2.0 + 0.25 * torch.randn(512, 1)
eps = 0.25 * torch.randn(512, 1)          # fixed noise for the fake cloud

def make_net():
    return nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(),
                         nn.Linear(64, 1))

def bce_gradient(mu_val):
    """Train a BCE discriminator to convergence, then measure the generator's
    gradient d/dmu of the non-saturating loss -log D(fake)."""
    torch.manual_seed(0)
    D = make_net()
    opt = torch.optim.Adam(D.parameters(), lr=1e-3)
    bce = nn.BCEWithLogitsLoss()
    fake = mu_val + eps
    for _ in range(500):
        loss = bce(D(real), torch.ones(512, 1)) + bce(D(fake), torch.zeros(512, 1))
        opt.zero_grad(); loss.backward(); opt.step()
    # saturating (original minimax) G loss: minimize log(1 - D(fake))
    mu = torch.tensor(mu_val, requires_grad=True)
    sat = torch.log(1 - torch.sigmoid(D(mu + eps)) + 1e-12).mean()
    sat.backward()
    g_sat = mu.grad.abs().item()
    # non-saturating G loss: minimize -log D(fake)
    mu = torch.tensor(mu_val, requires_grad=True)
    bce(D(mu + eps), torch.ones(512, 1)).backward()
    g_nonsat = mu.grad.abs().item()
    with torch.no_grad():
        p_fake = torch.sigmoid(D(mu_val + eps)).mean().item()
    return p_fake, g_sat, g_nonsat

def wgan_gradient(mu_val):
    """Train a critic with gradient penalty, then measure d/dmu of -C(fake)."""
    torch.manual_seed(0)
    C = make_net()
    opt = torch.optim.Adam(C.parameters(), lr=1e-3)
    fake = mu_val + eps
    for _ in range(500):
        a = torch.rand(512, 1)
        x_hat = (a * real + (1 - a) * fake).requires_grad_(True)
        grad = torch.autograd.grad(C(x_hat).sum(), x_hat, create_graph=True)[0]
        gp = ((grad.norm(2, dim=1) - 1) ** 2).mean()
        loss = C(fake).mean() - C(real).mean() + 10.0 * gp
        opt.zero_grad(); loss.backward(); opt.step()
    mu = torch.tensor(mu_val, requires_grad=True)
    (-C(mu + eps).mean()).backward()
    return mu.grad.abs().item()

print(f"{'mu':>6} {'distance':>9} {'D(fake)':>8} {'|sat grad|':>11} "
      f"{'|non-sat grad|':>15} {'|W grad|':>9}")
for mu_val in [1.5, 0.0, -2.0, -5.0, -8.0]:
    p, g_sat, g_nonsat = bce_gradient(mu_val)
    g_w = wgan_gradient(mu_val)
    print(f"{mu_val:>6.1f} {2.0 - mu_val:>9.1f} {p:>8.4f} {g_sat:>11.6f} "
          f"{g_nonsat:>15.4f} {g_w:>9.4f}")

## Constraining the critic: spectral normalization


*Expected output starts with:* `plain D          near -2: 0.471  near +2: 0.434  layer sigma_max ['3.58', '2.03', '0.97'`


In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def real_batch(n):
    side = (torch.rand(n, 1) < 0.5).float() * 4 - 2
    return side + 0.5 * torch.randn(n, 1)

def run(spectral, steps=2000):
    torch.manual_seed(0)
    G = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(),
                      nn.Linear(32, 1))
    layers = [nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(),
              nn.Linear(32, 1)]
    if spectral:
        layers = [nn.utils.spectral_norm(l) if isinstance(l, nn.Linear) else l
                  for l in layers]
    D = nn.Sequential(*layers)
    opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()
    ones, zeros = torch.ones(128, 1), torch.zeros(128, 1)
    for step in range(steps):
        x_real = real_batch(128)
        x_fake = G(torch.randn(128, 1)).detach()
        d_loss = bce(D(x_real), ones) + bce(D(x_fake), zeros)
        opt_d.zero_grad(); d_loss.backward(); opt_d.step()
        x_fake = G(torch.randn(128, 1))
        g_loss = bce(D(x_fake), ones)
        opt_g.zero_grad(); g_loss.backward(); opt_g.step()

    with torch.no_grad():
        s = G(torch.randn(4000, 1))
    left = ((s + 2).abs() < 1).float().mean().item()
    right = ((s - 2).abs() < 1).float().mean().item()
    # top singular value of each linear layer = its Lipschitz constant
    svs = [torch.linalg.svdvals(m.weight.detach())[0].item()
           for m in D.modules() if isinstance(m, nn.Linear)]
    # steepest slope of D over a dense grid of inputs
    x = torch.linspace(-6, 6, 2001).unsqueeze(1).requires_grad_(True)
    grad = torch.autograd.grad(D(x).sum(), x)[0]
    return left, right, svs, grad.abs().max().item()

for spectral in [False, True]:
    left, right, svs, gmax = run(spectral)
    name = "spectral-norm D" if spectral else "plain D        "
    print(f"{name}  near -2: {left:.3f}  near +2: {right:.3f}  "
          f"layer sigma_max {[f'{v:.2f}' for v in svs]}  max |dD/dx| {gmax:.3f}")

## Try it yourself

1. Rerun the 1D GAN with seeds 1 through 4. Does mode collapse happen in every run? Onto which mode, and does every run recover by step 2000? Summarize coverage at steps 250 and 2000 across seeds.
2. Raise both learning rates from `2e-4` to `2e-3` and rerun. Describe what happens to the loss traces and the coverage numbers — this is the cheapest instability demonstration available.
3. In the 8-mode experiment, train the discriminator 3 times per generator step. Does coverage develop faster, slower, or fail? Compare final per-mode counts.
4. Add one line implementing one-sided label smoothing (real labels 0.9 instead of 1.0) in both scripts and compare the final coverage statistics against the originals.
5. In the gradient-vanishing experiment, reduce the discriminator training from 500 steps to 50 and rebuild the table. How does the saturating-loss column change, and why would an *undertrained* discriminator hide the pathology? Relate your answer to the folklore advice of keeping `D` weak.
6. Wrap the *generator* in `spectral_norm` instead of the discriminator and rerun the coverage experiment. Report what happens, then reason about the asymmetry: what does a Lipschitz constraint mean for a network that must map a unit Gaussian onto modes at distance 2, versus for a critic that only scores inputs?


---

Full discussion of everything above: [S24 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s24/).
